In [1]:

import pandas as pd
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import os

In [2]:
def create_spatial_model(file_path='CGWB_data_main_cleaned.csv'):
    """
    Loads data, trains a spatial model (KNN), and returns it along with the long-format dataframe.
    """
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Error: {file_path} not found. Cannot create the spatial model.")
        return None, None

    # Melt to long format
    id_vars = ['STATE', 'DISTRICT', 'LAT', 'LON', 'SITE_TYPE', 'WLCODE']
    df_long = pd.melt(df, id_vars=id_vars, var_name='Date', value_name='Water_Level')

    # Convert 'Date' and drop missing
    df_long['Date'] = pd.to_datetime(df_long['Date'], format='%d/%m/%Y', errors='coerce')
    df_long.dropna(subset=['Water_Level', 'LAT', 'LON', 'Date'], inplace=True)

    # Most recent values for spatial KNN
    df_recent = df_long.loc[df_long.groupby('WLCODE')['Date'].idxmax()]

    # Features and target
    X = df_recent[['LAT', 'LON']]
    y = df_recent['Water_Level']

    # Train KNN model
    knn_model = KNeighborsRegressor(n_neighbors=5, weights='distance')
    knn_model.fit(X, y)

    print("✅ Spatial model trained successfully.")
    return knn_model, df_long

In [3]:
def predict_until_year(latitude, longitude, future_year, model, df_long, output_dir="outputs"):
    """
    Predicts groundwater levels from the last available year up to future_year
    for a given location using:
    1. KNN to find nearest well
    2. Linear Regression on historical data
    Also saves:
      - Prediction plot as PNG
      - Prediction data as CSV
    """
    if model is None or df_long is None:
        return "Model not available."

    # Step 1: Find nearest well
    input_data = pd.DataFrame([[latitude, longitude]], columns=['LAT', 'LON'])
    distances, indices = model.kneighbors(input_data, n_neighbors=1)
    
    # Get nearest well code
    nearest_point = model._fit_X[indices[0][0]]
    nearest_lat, nearest_lon = nearest_point
    nearest_well = df_long[
        (df_long['LAT'] == nearest_lat) & (df_long['LON'] == nearest_lon)
    ]['WLCODE'].iloc[0]

    # Step 2: Extract historical data
    ts = df_long[df_long['WLCODE'] == nearest_well].copy()
    ts['Year'] = ts['Date'].dt.year
    yearly_avg = ts.groupby('Year')['Water_Level'].mean().reset_index()

    # Step 3: Train Linear Regression
    X = yearly_avg[['Year']]
    y = yearly_avg['Water_Level']
    lr = LinearRegression()
    lr.fit(X, y)

    # Step 4: Predict from last available year to future_year
    last_year = yearly_avg['Year'].max()
    prediction_years = list(range(last_year + 1, future_year + 1))
    predictions = lr.predict([[yr] for yr in prediction_years])

    # Pack results
    results = dict(zip(prediction_years, predictions))

    # --- Create output directory ---
    os.makedirs(output_dir, exist_ok=True)

    # Step 5: Visualization & Save Plot
    plt.figure(figsize=(8, 5))
    plt.plot(yearly_avg['Year'], yearly_avg['Water_Level'], marker='o', label="Historical Data")
    plt.plot(prediction_years, predictions, marker='o', linestyle='--', color='red', label="Predicted Future")
    plt.axvline(x=last_year, color='gray', linestyle=':', label="Forecast Start")
    plt.xlabel("Year")
    plt.ylabel("Groundwater Level (m)")
    plt.title(f"Groundwater Level Prediction for Well {nearest_well}")
    plt.legend()
    plt.grid(True)

    plot_path = os.path.join(output_dir, f"prediction_{nearest_well}.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📊 Prediction plot saved as {plot_path}")

    # Step 6: Save Historical + Predicted Data as CSV
    pred_df = pd.DataFrame({
        "Year": list(yearly_avg['Year']) + prediction_years,
        "Water_Level": list(yearly_avg['Water_Level']) + list(predictions),
        "Type": ["Historical"] * len(yearly_avg) + ["Predicted"] * len(prediction_years)
    })

    csv_path = os.path.join(output_dir, f"prediction_{nearest_well}.csv")
    pred_df.to_csv(csv_path, index=False)
    print(f"📂 Prediction data saved as {csv_path}")

    return nearest_well, results, plot_path, csv_path

In [4]:
spatial_model, df_long = create_spatial_model('CGWB_data_cleaned.csv')

if spatial_model:
    input_lat = float(input("Enter latitude: "))
    input_lon = float(input("Enter longitude: "))
    input_year = int(input("Enter future year to predict up to (e.g. 2030): "))

    well, future_preds, plot_file, csv_file = predict_until_year(
        input_lat, input_lon, input_year, spatial_model, df_long
    )

    print(f"\n--- Year-wise Future Groundwater Prediction ---")
    print(f"Nearest well: {well}")
    print(f"Location: ({input_lat}, {input_lon})\n")

    for yr, level in future_preds.items():
        print(f"Predicted groundwater level in {yr}: {level:.2f} meters")


✅ Spatial model trained successfully.


c:\Users\mudit\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


📊 Prediction plot saved as outputs\prediction_W30848.png
📂 Prediction data saved as outputs\prediction_W30848.csv

--- Year-wise Future Groundwater Prediction ---
Nearest well: W30848
Location: (78.45, 28.45)

Predicted groundwater level in 2018: 2.74 meters
Predicted groundwater level in 2019: 2.81 meters
Predicted groundwater level in 2020: 2.87 meters
Predicted groundwater level in 2021: 2.93 meters
Predicted groundwater level in 2022: 3.00 meters
Predicted groundwater level in 2023: 3.06 meters
Predicted groundwater level in 2024: 3.12 meters
Predicted groundwater level in 2025: 3.18 meters
Predicted groundwater level in 2026: 3.25 meters
Predicted groundwater level in 2027: 3.31 meters
Predicted groundwater level in 2028: 3.37 meters
Predicted groundwater level in 2029: 3.44 meters
Predicted groundwater level in 2030: 3.50 meters
Predicted groundwater level in 2031: 3.56 meters
Predicted groundwater level in 2032: 3.62 meters
Predicted groundwater level in 2033: 3.69 meters
Predic